# STEP 08b - 직장/방문/서비스/토지이용
**08b 완료 후 → COMPAS_08c 실행**

In [ ]:
import os

BASE_DIR   = os.getcwd()          # /opt/app-root/src
EPDO_DIR   = os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(EPDO_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR : {BASE_DIR}")
print(f"DATA_DIR : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


In [ ]:
import csv
import json
import os
import warnings
from collections import defaultdict, Counter

import geopandas as gpd
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)   # CRS 관련 경고 억제


# 입력
GAP_FILE     = os.path.join(OUTPUT_DIR, "06_인프라공백_고위험격자.csv")
STEP07_FILE  = os.path.join(OUTPUT_DIR, "07_링크_속도혼잡_보강.csv")
GRID_FILE    = os.path.join(BASE_DIR, "data", "01._격자_(4개_시·구).geojson")
RES_FILE     = os.path.join(BASE_DIR, "data", "03._성연령별_거주인구(격자).csv")
FLOAT_FILE   = os.path.join(BASE_DIR, "data", "04._성연령별_유동인구.csv")
WORK_FILE    = os.path.join(BASE_DIR, "data", "05._시간대별_직장인구.csv")
VISIT_FILE   = os.path.join(BASE_DIR, "data", "06._시간대별_방문인구.csv")
SVC_FILE     = os.path.join(BASE_DIR, "data", "07._주중주말_서비스인구.csv")
ROAD_FILE    = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
LU_FILE      = os.path.join(BASE_DIR, "data", "22._토지이용계획도_(4개_신도시).geojson")
LU_HANAM     = os.path.join(BASE_DIR, "data", "23._토지이용계획도_(하남교산).geojson")

# 출력
OUTPUT_PATH  = os.path.join(OUTPUT_DIR, "08_격자_종합위험지수.csv")
HANAM_OUTPUT = os.path.join(OUTPUT_DIR, "08_하남교산_예측위험도.csv")

CRS_PROJ     = "EPSG:5186"   # 한국 중부원점 (단위: 미터, centroid 계산 정확)
CRS_GEO      = "EPSG:4326"   # WGS84

# 교통약자 취약 시간대: 등교(07~09), 하교(13~16), 퇴근(17~19)
VULN_SLOTS   = {7, 8, 9, 13, 14, 15, 16, 17, 18, 19}
SLOT_COLS    = [f"TMST_{str(h).zfill(2)}" for h in VULN_SLOTS]
ALL_SLOTS    = [f"TMST_{str(h).zfill(2)}" for h in range(24)]

# 토지이용 → 위험 카테고리 매핑
SCHOOL_TYPES    = {"학교", "초등학교", "중학교", "고등학교", "교육시설"}
RESIDENT_TYPES  = {"아파트", "연립주택", "다세대주택", "단독주택", "공동주택", "주상복합", "연립"}
COMMERCIAL_TYPES = {"상업", "일반상업", "근린상업", "근린생활시설용지", "상업시설"}


# ── 유틸 ────────────────────────────────────────────────────────────────────

def load_csv_dict(path, key_col):
    with open(path, "r", encoding="utf-8-sig") as f:
        return {row[key_col]: row for row in csv.DictReader(f)}


def safe_float(val, default=0.0):
    try:
        return float(val or 0)
    except (ValueError, TypeError):
        return default


def points_to_grid(lon_col, lat_col, rows, value_cols, grid_gdf):
    """lon/lat 포인트 → 격자 공간 join 후 격자별 집계 반환"""
    pts = []
    for row in rows:
        try:
            lon, lat = float(row[lon_col]), float(row[lat_col])
        except (ValueError, KeyError):
            continue
        rec = {"geometry": Point(lon, lat)}
        for c in value_cols:
            rec[c] = safe_float(row.get(c))
        pts.append(rec)

    pts_gdf  = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    joined   = gpd.sjoin(pts_gdf, grid_gdf[["gid", "geometry"]], how="left", predicate="within")
    return joined


def median(values):
    sv = sorted(v for v in values if v is not None)
    if not sv:
        return 0
    mid = len(sv) // 2
    return (sv[mid - 1] + sv[mid]) / 2 if len(sv) % 2 == 0 else sv[mid]


# ── 메인 ─────────────────────────────────────────────────────────────────────


## ── 1. 고위험 격자 목록

In [ ]:
print("\n[1] 고위험 격자 로드 중...")
gap_data = load_csv_dict(GAP_FILE, "grid_gid")
print(f"    고위험 격자 수: {len(gap_data):,}개")


## ── 2. 격자 GeoDataFrame (고위험만 필터)

In [ ]:
print("\n[2] 격자 폴리곤 로드 중...")
grid_gdf = gpd.read_file(GRID_FILE)
grid_gdf = grid_gdf[grid_gdf["gid"].isin(gap_data.keys())].reset_index(drop=True)
print(f"    고위험 격자 폴리곤: {len(grid_gdf):,}개")


In [ ]:
import json as _j, os as _os
_tmp = os.path.join(OUTPUT_DIR, "_tmp_step08")
def _load(n):
    with open(_os.path.join(_tmp, f"{n}.json"), encoding="utf-8") as _f:
        return _j.load(_f)
speed_agg = _load("speed_agg")
res_data  = _load("res_data")
float_agg = _load("float_agg")
print(f"로드 완료: speed={len(speed_agg)}, res={len(res_data)}, float={len(float_agg)}")


## ── 6. 05번 직장인구 — 취약 시간대 공간 join

In [ ]:
print("[6] 직장인구(취약 시간대) 공간 join 중... (청크 처리)")
import gc
from collections import defaultdict as _dd
from shapely.geometry import Point as _Pt
CHUNK = 30000
_work_raw = _dd(lambda: {"vuln_work": 0.0, "total_work": 0.0})
def _proc_work(rows, agg):
    pts = [{"geometry": _Pt(float(r["lon"]), float(r["lat"])), **{c: safe_float(r.get(c)) for c in SLOT_COLS+ALL_SLOTS}}
           for r in rows if r.get("lon") and r.get("lat")]
    if not pts: return
    _g = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    _j2 = gpd.sjoin(_g, grid_gdf[["gid","geometry"]], how="left", predicate="within")
    for gid, grp in _j2.dropna(subset=["gid"]).groupby("gid"):
        agg[str(gid)]["vuln_work"]  += sum(grp[c].sum() for c in SLOT_COLS if c in grp.columns)
        agg[str(gid)]["total_work"] += sum(grp[c].sum() for c in ALL_SLOTS if c in grp.columns)
    del _g, _j2; gc.collect()
with open(WORK_FILE, "r", encoding="utf-8-sig") as _f:
    _rdr = csv.DictReader(_f); _ch = []
    for _row in _rdr:
        _ch.append(_row)
        if len(_ch) >= CHUNK: _proc_work(_ch, _work_raw); _ch = []
    if _ch: _proc_work(_ch, _work_raw)
work_agg = {g: {"vuln_work": round(v["vuln_work"],4), "total_work": round(v["total_work"],4)} for g,v in _work_raw.items()}
print(f"    직장인구 매칭 격자: {len(work_agg):,}개")
gc.collect()


## ── 7. 06번 방문인구 — 취약 시간대 공간 join

In [ ]:
print("[7] 방문인구(취약 시간대) 공간 join 중... (청크 처리)")
import gc
from collections import defaultdict as _dd
from shapely.geometry import Point as _Pt
CHUNK = 30000
_visit_raw = _dd(lambda: {"vuln_visit": 0.0, "total_visit": 0.0})
def _proc_visit(rows, agg):
    pts = [{"geometry": _Pt(float(r["lon"]), float(r["lat"])), **{c: safe_float(r.get(c)) for c in SLOT_COLS+ALL_SLOTS}}
           for r in rows if r.get("lon") and r.get("lat")]
    if not pts: return
    _g = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    _j2 = gpd.sjoin(_g, grid_gdf[["gid","geometry"]], how="left", predicate="within")
    for gid, grp in _j2.dropna(subset=["gid"]).groupby("gid"):
        agg[str(gid)]["vuln_visit"]  += sum(grp[c].sum() for c in SLOT_COLS if c in grp.columns)
        agg[str(gid)]["total_visit"] += sum(grp[c].sum() for c in ALL_SLOTS if c in grp.columns)
    del _g, _j2; gc.collect()
with open(VISIT_FILE, "r", encoding="utf-8-sig") as _f:
    _rdr = csv.DictReader(_f); _ch = []
    for _row in _rdr:
        _ch.append(_row)
        if len(_ch) >= CHUNK: _proc_visit(_ch, _visit_raw); _ch = []
    if _ch: _proc_visit(_ch, _visit_raw)
visit_agg = {g: {"vuln_visit": round(v["vuln_visit"],4), "total_visit": round(v["total_visit"],4)} for g,v in _visit_raw.items()}
print(f"    방문인구 매칭 격자: {len(visit_agg):,}개")
gc.collect()


## ── 8. 07번 서비스인구 — 주중/주말 분리 공간 join

In [ ]:
print("\n[8] 서비스인구(주중/주말) 공간 join 중...")
svc_h, svc_w = [], []
with open(SVC_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        (svc_h if row.get("hw") == "H" else svc_w).append(row)

def agg_svc(rows, prefix):
    joined = points_to_grid("lon", "lat", rows, ["w_pop", "v_pop"], grid_gdf)
    agg = {}
    for gid, grp in joined.dropna(subset=["gid"]).groupby("gid"):
        agg[str(gid)] = {
            f"{prefix}_work":  round(grp["w_pop"].sum(), 4),
            f"{prefix}_visit": round(grp["v_pop"].sum(), 4),
        }
    return agg

wd_agg = agg_svc(svc_h, "weekday")
we_agg = agg_svc(svc_w, "weekend")
print(f"    주중 매칭: {len(wd_agg):,}개 | 주말 매칭: {len(we_agg):,}개")


## ── 9. 22번 토지이용 — 수정 2: intersects로 매칭률 개선

In [ ]:
print("\n[9] 토지이용(4개 신도시) 공간 join 중...")
lu_gdf = gpd.read_file(LU_FILE)

# within → intersects: 격자 폴리곤과 토지이용 폴리곤이 조금이라도 겹치면 매칭
lu_joined = gpd.sjoin(
    grid_gdf[["gid", "geometry"]],
    lu_gdf[["blockType", "geometry"]],
    how="left",
    predicate="intersects"
)
lu_map = {}
for gid, grp in lu_joined.dropna(subset=["blockType"]).groupby("gid"):
    lu_map[str(gid)] = grp["blockType"].value_counts().idxmax()
print(f"    토지이용 매칭 격자: {len(lu_map):,}개 "
      f"({len(lu_map)/len(gap_data)*100:.1f}%)")


In [ ]:
import json as _j, os as _os
_tmp = os.path.join(OUTPUT_DIR, "_tmp_step08")
for _name, _obj in [
    ("work_agg",  work_agg),
    ("visit_agg", visit_agg),
    ("wd_agg",    wd_agg),
    ("we_agg",    we_agg),
    ("lu_map",    lu_map),
]:
    with open(_os.path.join(_tmp, f"{_name}.json"), "w", encoding="utf-8") as _f:
        _j.dump(_obj, _f, ensure_ascii=False)
print("저장 완료 → COMPAS_08c 실행")
